In [5]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict
import ot  # POT: Python Optimal Transport
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import maxflow
from sklearn.cluster import KMeans

In [6]:
csv_path = "shubert_pose_clusters.csv"
embedding_dir = "output_embeddings"

# Load CSV
df = pd.read_csv(csv_path)

# Initialize dictionary to store embeddings grouped by cluster
cluster_embeddings = defaultdict(list)

for _, row in df.iterrows():
    cluster_id = row['cluster_id']
    video_file = row['video']  # e.g., Sk1Vj0ndA_w-261.npy
    embed_path = os.path.join(embedding_dir, video_file)

    if os.path.exists(embed_path):
        emb = np.load(embed_path)  # Expected shape: (T, 768)

        if emb.ndim == 3 and emb.shape[0] == 12:
            emb = np.mean(emb, axis=0)  # SHuBERT stack: (12, T, 768) → (T, 768)

        if emb.ndim == 2 and emb.shape[1] == 768:
            cluster_embeddings[cluster_id].append(emb)
        else:
            print(f"Skipping {video_file} due to unexpected shape: {emb.shape}")
    else:
        print(f"Missing embedding file: {embed_path}")

print("Loaded cluster-wise video embeddings:")
for cid, vids in cluster_embeddings.items():
    print(f" - Cluster {cid}: {len(vids)} videos")

Loaded cluster-wise video embeddings:
 - Cluster 0: 4 videos
 - Cluster 1: 6 videos


In [7]:
def compute_ot_alignment(x, y, epsilon=0.05, temporal_prior_weight=0.1):
    T_x, D = x.shape
    T_y, _ = y.shape

    # Cost matrix (squared Euclidean)
    M = np.linalg.norm(x[:, None, :] - y[None, :, :], axis=2) ** 2

    # Temporal prior
    t_x_norm = np.linspace(0, 1, T_x)
    t_y_norm = np.linspace(0, 1, T_y)
    temporal_dist = np.abs(t_x_norm[:, None] - t_y_norm[None, :])
    temporal_prior = np.exp(-temporal_prior_weight * temporal_dist)

    cost_matrix = M / M.max()
    cost_matrix = cost_matrix / temporal_prior

    # Uniform marginals
    a = np.ones(T_x) / T_x
    b = np.ones(T_y) / T_y

    # Sinkhorn transport plan
    T = ot.sinkhorn(a, b, cost_matrix, reg=epsilon)
    return T

In [8]:
def compute_similarity_matrix(cluster_embs):
    N = sum([e.shape[0] for e in cluster_embs])
    W = np.zeros((N, N))
    offset = 0
    index_map = []

    for i, xi in enumerate(cluster_embs):
        ti = xi.shape[0]
        for j, xj in enumerate(cluster_embs):
            tj = xj.shape[0]
            T = compute_ot_alignment(xi, xj)

            # Normalize and store OT matrix in full graph
            si = sum(e.shape[0] for e in cluster_embs[:i])
            sj = sum(e.shape[0] for e in cluster_embs[:j])
            W[si:si+ti, sj:sj+tj] += T

        for k in range(ti):
            index_map.append((i, k))  # (video_id, frame_id)
        offset += ti

    W = (W + W.T) / 2  # Symmetrize
    return W, index_map

In [9]:
def run_graph_cut_segmentation(embeddings, similarity_matrix, n_steps, lambda_pairwise=10.0):
    N = embeddings.shape[0]
    # g = maxflow.Graph[float]()
    g = maxflow.GraphFloat()
    nodes = g.add_nodes(N)

    # Unary potentials from KMeans distances
    kmeans = KMeans(n_clusters=n_steps).fit(embeddings)
    centers = kmeans.cluster_centers_
    unary_costs = np.zeros((N, n_steps))

    for i in range(N):
        for l in range(n_steps):
            unary_costs[i, l] = np.linalg.norm(embeddings[i] - centers[l]) ** 2

    # Simplified graph cut using only label 0 vs others
    for i in range(N):
        g.add_tedge(nodes[i], unary_costs[i, 0], np.min(unary_costs[i, 1:]))

    for i in range(N):
        for j in range(i + 1, N):
            sim = similarity_matrix[i, j]
            if sim > 0.5:
                cost = lambda_pairwise * (1 - sim)
                g.add_edge(nodes[i], nodes[j], cost, cost)

    g.maxflow()
    binary_labels = np.array([g.get_segment(n) for n in nodes])

    # Reassign with KMeans on selected frames
    key_frames = embeddings[binary_labels == 0]
    key_labels = KMeans(n_clusters=n_steps).fit_predict(key_frames)

    final_labels = np.full(N, -1)
    final_labels[binary_labels == 0] = key_labels
    return final_labels

In [10]:
def order_keysteps_by_time(labels, index_map):
    keystep_timestamps = defaultdict(list)

    for idx, label in enumerate(labels):
        if label == -1:
            continue
        video_id, frame_id = index_map[idx]
        norm_time = frame_id / 100  # approximate, or use actual T
        keystep_timestamps[label].append(norm_time)

    label_avg_times = {k: np.mean(v) for k, v in keystep_timestamps.items()}
    ordered_labels = [label for label, _ in sorted(label_avg_times.items(), key=lambda x: x[1])]
    return ordered_labels

In [11]:
# ✅ CELL 7: Run Full Pipeline per Cluster
all_cluster_keystep_labels = {}
all_cluster_orders = {}

for cluster_id, embs in cluster_embeddings.items():
    print(f"\nProcessing cluster {cluster_id} with {len(embs)} videos...")

    all_embs = np.concatenate(embs, axis=0)
    W, index_map = compute_similarity_matrix(embs)
    labels = run_graph_cut_segmentation(all_embs, W, n_steps=5)
    order = order_keysteps_by_time(labels, index_map)

    all_cluster_keystep_labels[cluster_id] = labels
    all_cluster_orders[cluster_id] = order

    print(f" - Ordered Key-Steps: {order}")


Processing cluster 0 with 4 videos...
 - Ordered Key-Steps: [np.int64(0), np.int64(4), np.int64(1), np.int64(3), np.int64(2)]

Processing cluster 1 with 6 videos...


/DATA5/ashishu23/SURGE/.conda/lib/python3.9/site-packages/ot/bregman/_sinkhorn.py:667: UserWarning: Sinkhorn did not converge. You might want to increase the number of iterations `numItermax` or the regularization parameter `reg`.
  warnings.warn(


 - Ordered Key-Steps: [np.int64(2), np.int64(0), np.int64(4), np.int64(3), np.int64(1)]


In [13]:
import pickle

# map from cluster to filenames
cluster_video_map = {cid: [os.path.basename(f) for f in df[df['cluster_id'] == cid]['video']] 
                     for cid in cluster_embeddings}

# Save everything needed for Translation
with open("opel_outputs.pkl", "wb") as f:
    pickle.dump({
        "keystep_labels": all_cluster_keystep_labels,
        "keystep_orders": all_cluster_orders,
        "cluster_video_map": cluster_video_map
    }, f)

print("✅ Saved OPEL output.")


✅ Saved OPEL output.
